<a href="https://colab.research.google.com/github/TobiasNeumeier/ProteinPrediction/blob/main/protein_prediction_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests

url = "https://www.ebi.ac.uk/interpro/api/protein/reviewed/entry/pfam/"
response = requests.get(url, headers={"Accept": "application/json"})

if response.ok:
    data = response.json()
    print("Number of entries:", len(data['results']))
    for entry in data['results'][:5]:  # Print first 5 entries
        print(entry['metadata']['accession'], entry['metadata']['name'])
else:
    print("Failed to fetch data:", response.status_code)


In [ ]:
data

In [ ]:
import requests

# Example UniProt accession
accession = "A0A017SE81"

url = f"https://www.ebi.ac.uk/interpro/api/protein/uniprot/{accession}/"

headers = {
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)

if response.ok:
    entry = response.json()
    print("Entry:", entry['metadata']['accession'], "-", entry['metadata']['source_database'])
    print("Type:", entry['metadata'].get('entry_type', 'N/A'))
    print("Description:", entry['metadata'].get('description', 'N/A'))
    print()
else:
    print("Request failed:", response.status_code)


In [ ]:
entry["metadata"].keys()

In [ ]:
entry["metadata"]["sequence"]

In [ ]:
import requests

def fetch_sequence(accession):
    url = f"https://www.ebi.ac.uk/interpro/api/protein/uniprot/{accession}/"

    headers = {
        "Accept": "application/json"
    }

    response = requests.get(url, headers=headers)

    if response.ok:
        entry = response.json()
        return entry["metadata"]["sequence"]
    else:
        print("Request failed:", response.status_code)
        return None


url = "https://www.ebi.ac.uk/interpro/api/protein/reviewed/entry/pfam/"
response = requests.get(url, headers={"Accept": "application/json"})
accessions = {}

if response.ok:
    data = response.json()
    print("Number of entries:", len(data['results']))
    for entry in data['results'][:5]:
        sequence = fetch_sequence(entry["metadata"]["accession"])
        accessions[entry['metadata']['accession']] = sequence
else:
    print("Failed to fetch data:", response.status_code)

In [ ]:
accessions

In [ ]:
import requests

uniprot_acc = "P00533"
url = f"https://rest.uniprot.org/uniprotkb/{uniprot_acc}.fasta"

response = requests.get(url)
if response.ok:
    fasta = response.text
    print(fasta)
else:
    print("Failed to fetch sequence:", response.status_code)


In [ ]:
url = f"https://www.ebi.ac.uk/interpro/api/protein/uniprot/{uniprot_acc}/entry_interpro/"

headers = {"Accept": "application/json"}
response = requests.get(url, headers=headers)

if response.ok:
    result = response.json()
    entry = result['metadata']
    print("Domain:", entry['accession'], entry['description'])
    print(entry.keys())
else:
    print("Failed to fetch domain annotations:", response.status_code)


In [ ]:
!wget https://ftp.ebi.ac.uk/pub/databases/Pfam/current_release/Pfam-A.seed.gz
!gunzip Pfam-A.seed.gz  # decompress

In [ ]:
!head -n 40 Pfam-A.seed

In [ ]:
from Bio import AlignIO

alignments = AlignIO.parse("Pfam-A.seed", "stockholm")

for i, aln in enumerate(alignments):
    print(f"Alignment #{i + 1}, {len(aln)} sequences, family: {aln.annotations.get('PFAM', '')}")
    print(aln[0].id, aln[0].seq[:50], "...")
    if i == 2:  # only print first 3 alignments
        break

In [ ]:
# Install Biopython

!pip install biopython

In [ ]:
import requests

def get_pfam_domains(uniprot_id):
    url = f"https://www.ebi.ac.uk/interpro/api/protein/uniprot/{uniprot_id}"
    response = requests.get(url, headers={"Accept": "application/json"})
    if response.status_code == 200:
        data = response.json()
        for entry in data.get('entries', []):
            if entry['entry_db'] == 'Pfam':
                print(f"Pfam ID: {entry['entry_ac']}, Description: {entry['entry_name']}")
        return data
    else:
        print(f"Failed to retrieve data for {uniprot_id}")

# Example usage
data = get_pfam_domains("P12345")

In [ ]:
data

In [ ]:
import requests

def get_families():
    url = "https://www.ebi.ac.uk/interpro/api/entry/pfam/?type=family"
    response = requests.get(url, headers={"Accept": "application/json"})
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Failed to retrieve data. Status code: {response.status_code}")
        return None

data = get_families()

In [ ]:
data

In [ ]:
for entry in data["results"][:20]:
    print(f"Accession: {entry['metadata']['accession']}")

In [ ]:
url = "https://www.ebi.ac.uk/interpro/api/entry/pfam/PF00012"

response = requests.get(url, headers={"Accept": "application/json"})
if response.status_code == 200:
    family_entry = response.json()

In [ ]:
family_entry

In [ ]:
def fetch_uniprot_sequences(pfam_id):
    # Search UniProt for proteins in a specific Pfam family
    url = f"https://www.uniprot.org/uniprot/?query=pfam:{pfam_id}&format=fasta"
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        return None

# Fetch sequences for a given Pfam ID
sequences = fetch_uniprot_sequences("PF00012")
print(sequences)  # Inspect the sequences


In [ ]:
# achtung alles von chatty
import os
import json
import requests
from pathlib import Path
from tqdm import tqdm
from Bio import SeqIO
from transformers import T5Tokenizer, T5EncoderModel
import torch

# Config
PFAM_IDS = ['PF00005', 'PF00069']  # Example
RAW_DIR = Path("data/raw")
PROC_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

def get_pfam_entries(pfam_id, max_proteins=20):
    url = f"https://www.ebi.ac.uk/interpro/api/entry/pfam/{pfam_id}/protein/entry_protein_location/"
    headers = {'Accept': 'application/json'}
    proteins = []
    while url and len(proteins) < max_proteins:
        r = requests.get(url, headers=headers)
        if not r.ok:
            break
        data = r.json()
        proteins.extend(data['results'])
        url = data.get('next')
    return proteins[:max_proteins]

def download_uniprot_sequence(uniprot_ac):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_ac}.fasta"
    r = requests.get(url)
    if r.ok:
        return str(r.text)
    return None

def save_sequences_and_domains(families):
    seq_records = {}
    domain_info = {}
    for pfam_id in families:
        entries = get_pfam_entries(pfam_id)
        for entry in tqdm(entries, desc=f"Processing {pfam_id}"):
            acc = entry['metadata']['accession']
            for protein in entry['entry_protein_locations']:
                uid = protein['protein']['accession']
                if uid not in seq_records:
                    seq = download_uniprot_sequence(uid)
                    if seq:
                        fasta_path = RAW_DIR / f"{uid}.fasta"
                        fasta_path.write_text(seq)
                        seq_records[uid] = seq
                for frag in protein['fragments']:
                    domain_info.setdefault(uid, []).append({
                        'pfam_id': pfam_id,
                        'start': frag['start'],
                        'end': frag['end']
                    })
    with open(RAW_DIR / "domain_mapping.json", "w") as f:
        json.dump(domain_info, f, indent=2)

def compute_prott5_embeddings():
    tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False)
    model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50")
    model.eval()
    model = model.to("cuda" if torch.cuda.is_available() else "cpu")

    for fasta_file in RAW_DIR.glob("*.fasta"):
        records = list(SeqIO.parse(fasta_file, "fasta"))
        if not records:
            continue
        seq = str(records[0].seq).replace(" ", "").replace("\n", "")
        ids = tokenizer.batch_encode_plus([seq], add_special_tokens=True, return_tensors="pt")
        ids = {k: v.to(model.device) for k, v in ids.items()}
        with torch.no_grad():
            emb = model(**ids).last_hidden_state.squeeze(0).cpu()
        torch.save(emb, PROC_DIR / f"{fasta_file.stem}_embedding.pt")

def label_residues_and_save():
    with open(RAW_DIR / "domain_mapping.json") as f:
        domain_map = json.load(f)

    for uid, domains in domain_map.items():
        embedding_path = PROC_DIR / f"{uid}_embedding.pt"
        if not embedding_path.exists():
            continue
        emb = torch.load(embedding_path)
        labels = torch.zeros(emb.shape[0], dtype=torch.long)
        for i, domain in enumerate(domains):
            labels[domain['start']:domain['end'] + 1] = i + 1  # Label domain with index
        torch.save(labels, PROC_DIR / f"{uid}_labels.pt")

# Pipeline
save_sequences_and_domains(PFAM_IDS)
#compute_prott5_embeddings()
#label_residues_and_save()


In [ ]:
import requests


def fetch_first_family_accession():
    # Endpoint to retrieve all Pfam entries
    pfam_url = "https://www.ebi.ac.uk/interpro/api/entry/pfam/?type=family"

    # Send GET request
    response = requests.get(pfam_url)

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()
        # Extract the first Pfam family accession
        first_family = data['results'][0]['metadata']['accession']
        return first_family
    else:
        print(f"Failed to retrieve Pfam families. Status code: {response.status_code}")


In [ ]:
def fetch_domain_occurences(family_accession):
    proteins_url = f"https://www.ebi.ac.uk/interpro/api/protein/UniProt/entry/pfam/{family_accession}/"
    response = requests.get(proteins_url)

    result = {}

    if response.status_code == 200:
        proteins_data = response.json()
        # Iterate through the proteins and extract accession, start, and end positions
        for protein in proteins_data["results"]:
            accession = protein['metadata']['accession']
            for entry in protein['entries'][0]['entry_protein_locations']:
                start = entry['fragments'][0]['start']
                end = entry['fragments'][0]['end']
                result[accession] = (start, end)
        return result
    else:
        print(f"Failed to retrieve proteins for family {family_accession}. Status code: {response.status_code}")

In [ ]:
def fetch_sequence(protein_acc):
    url = f"https://www.ebi.ac.uk/interpro/api/protein/uniprot/{protein_acc}/"
    response = requests.get(url)

    if response.status_code == 200:
        protein_info = response.json()
        sequence = protein_info["metadata"]["sequence"]
        return sequence
    else:
        print(f"Failed to retrieve information for protein {protein_accession}. Status code: {response.status_code}")

In [ ]:
family_acc = fetch_first_family_accession()
occ = fetch_domain_occurences(family_acc)

In [ ]:
occ

In [ ]:
result = [{'protein_accession': k, 'start': v[0], 'end': v[1], 'sequence': fetch_sequence(k)} for k,v in occ.items()]

In [ ]:
for v in result:
    print(v['sequence'] == result[0]['sequence'])

In [ ]:
ex = result[0]
start = result[0]['start']
end = result[0]['end']
len(ex['sequence'][start:end])

In [ ]:
len(ex['sequence'])

In [23]:
# Create mask
[family_acc  if (i >= start and i <= end) else 'x' for i, x in enumerate(ex['sequence'])]